# LLM Efficiency Frontier — analysis (v2)

Mirrors `src/analysis.py`. See README changelog for v2 fixes (interpolation clamping, exact log form, non-circular LOOCV comparison).

In [ ]:
import sys; sys.path.append('../src')
import numpy as np, pandas as pd, json
import analysis as A
A.DATA='../data/llm_scaling_dataset.csv'
A.DATA_EMP_FRONTIER='../data/llm_scaling_frontier.csv'
N,D,L = A.load()
print('points:', len(N), '| N range: %.2e - %.2e' % (N.min(), N.max()))

## 1. Full Chinchilla law L(N,D) — joint (N, D, C) space

In [ ]:
p = A.fit_ND(N,D,L)
pred = A.L_ND(N,D,p); r2 = 1-np.sum((L-pred)**2)/np.sum((L-L.mean())**2)
print('E=%.3f A=%.0f alpha=%.3f B=%.0f beta=%.3f | R2=%.4f' % (p['E'],p['A'],p['alpha'],p['B'],p['beta'],r2))

## 2. Compute-optimal frontier N*(C) = argmin_N L(N, C/6N)
Grid extends to 1e23.5 FLOP so the doubling gain g(N)=[L*(N)-L*(2N)]/L*(N) never leaves the grid (v2 fix).

In [ ]:
Nf,Lf = A.frontier(p)
lNf=np.log10(Nf)
for x in [1e8,1e9,1e10]: print(f'L*(N={x:.0e}) = {np.interp(np.log10(x),lNf,Lf):.3f}')

## 3. Marginal gain per doubling and hurdle-dependent threshold
Thresholds are flagged as interpolated vs. extrapolated w.r.t. the observed N range.

In [ ]:
ns = np.logspace(8,10.7,500)
g = A.gain_curve(Nf,Lf,ns)
for h in [0.02,0.04,0.06,0.08]:
    t = A.threshold(ns,g,h)
    tag = 'INTERPOLATED' if N.min()<=t<=N.max() else 'EXTRAPOLATED'
    print(f'hurdle {int(h*100)}%: N* = {t:.2e}  [{tag}]')

## 4. Model comparison — two ways
(a) On the model-derived frontier the power law is near-exact **by construction** (circular). (b) The honest test: fits + LOOCV on the 15 **empirical** envelope points.

In [ ]:
Ne,Le = A.load_empirical_frontier()
from scipy.optimize import curve_fit
ple,_ = curve_fit(A.log_model, Ne, Le, p0=A.LOG_P0, bounds=A.LOG_BOUNDS, maxfev=400000)
ppe,_ = curve_fit(A.power_law, Ne, Le, p0=A.POW_P0, maxfev=400000)
print('log  :', A.gof(A.log_model, ple, Ne, Le, 4), '| LOOCV RMSE %.4f' % A.loocv_rmse(A.log_model, A.LOG_P0, A.LOG_BOUNDS, Ne, Le))
print('power:', A.gof(A.power_law, ppe, Ne, Le, 3), '| LOOCV RMSE %.4f' % A.loocv_rmse(A.power_law, A.POW_P0, None, Ne, Le))

Over ~2.5 orders of magnitude the two forms are statistically indistinguishable on real data; the inner parameter *b* of the log form is unidentifiable. No superiority claim in either direction.

## 5. Monte Carlo robustness (seeded)
+/-2% loss noise -> refit -> frontier -> threshold@4%, 800 reps in the pipeline (200 here for speed). The median sits ~0.05 dex below the deterministic value (nonlinear-refit effect); both are reported.

In [ ]:
mc=[]
for _ in range(200):
    Lp = L*(1+A.RNG.normal(0,0.02,len(L)))
    p2 = A.fit_ND(N,D,Lp); nf,lf = A.frontier(p2)
    t = A.threshold(ns, A.gain_curve(nf,lf,ns), 0.04)
    if not np.isnan(t): mc.append(np.log10(t))
mc=np.array(mc)
print('median 10^%.3f | CI95 [10^%.3f, 10^%.3f]' % (np.median(mc), np.percentile(mc,2.5), np.percentile(mc,97.5)))